<a href="https://colab.research.google.com/github/Shakeel1111-creator/AI-ML-Classification-Project/blob/main/AI_ML_Classification_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

data = {
    "Age": [
        22,25,28,30,32,35,38,40,42,45,
        48,50,52,55,58,60,62,65,68,70,
        24,27,31,34,37,41,44,47,51,54
    ],

    "Monthly_Charges": [
        30,35,40,45,50,55,60,65,70,75,
        80,85,90,95,100,105,110,115,120,125,
        32,42,48,58,63,72,78,88,98,108
    ],

    "Tenure": [
        24,30,20,36,40,18,15,12,10,8,
        7,6,5,4,3,2,2,1,1,1,
        28,22,18,16,14,11,9,7,5,3
    ],

    "Support_Calls": [
        1,1,2,1,2,2,3,2,3,3,
        4,3,4,4,5,5,6,6,7,7,
        1,2,2,3,3,4,4,5,6,7
    ],

    "Contract": [
        "Long", "Long", "Long", "Long", "Long",
        "Long", "Long", "Long", "Long", "Long",
        "Short", "Short", "Short", "Short", "Short",
        "Short", "Short", "Short", "Short", "Short",
        "Long", "Long", "Long", "Long", "Short",
        "Short", "Short", "Short", "Short", "Short"
    ],

    "Churn": [
        0,0,0,0,0,0,0,0,0,0,
        0,0,0,0,0,0,0,1,1,1,
        0,0,0,0,0,1,1,1,1,1
    ]
}

df = pd.DataFrame(data)

X=df.drop("Churn",axis=1)
y=df["Churn"]

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

numerical_features=["Age","Monthly_Charges","Tenure","Support_Calls"]
categorial_features=["Contract"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
preprocessor=ColumnTransformer([
    ("num",StandardScaler(),numerical_features),
    ("cat",OneHotEncoder(),categorial_features)
])

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix

def train_and_evaluate(model):
  print(model)
  pipeline=Pipeline([
      ("preprocessor",preprocessor),
      ("smote",SMOTE(random_state=42,k_neighbors=2)),
      ("model",model)
  ])
  pipeline.fit(X_train,y_train)
  y_pred=pipeline.predict(X_test)
  accuracy=accuracy_score(y_test,y_pred)
  print("Accuracy: ",accuracy)
  precision=precision_score(y_test,y_pred)
  print("precision: ",precision)
  recall=recall_score(y_test,y_pred)
  print("recall: ",recall)
  f1=f1_score(y_test,y_pred)
  print("f1: ",f1)
  y_prob=pipeline.predict_proba(X_test)[:,1]
  auc=roc_auc_score(y_test,y_prob)
  print("AUC score is: ",auc)
  cm=confusion_matrix(y_test,y_pred)
  print("Confusion matrix: ",cm)
  print("--"*30)
  return pipeline,accuracy,precision,recall,f1,auc

results=[]
logistic_model,logistic_accuracy,logistic_precision,logistic_recall,logistic_f1,logistic_auc=train_and_evaluate(LogisticRegression())
results.append(["Logistic model",logistic_accuracy,logistic_precision,logistic_recall,logistic_f1,logistic_auc])

from sklearn.neighbors import KNeighborsClassifier
knn_model,knn_accuracy,knn_precision,knn_recall,knn_f1,knn_auc=train_and_evaluate(KNeighborsClassifier(n_neighbors=3))
results.append(["KNN model",knn_accuracy,knn_precision,knn_recall,knn_f1,knn_auc])

from sklearn.tree import DecisionTreeClassifier

tree_model,tree_accuracy,tree_precision,tree_recall,tree_f1,tree_auc=train_and_evaluate(DecisionTreeClassifier(random_state=42))
results.append(["Tree model",tree_accuracy,tree_precision,tree_recall,tree_f1,tree_auc])

from sklearn.ensemble import RandomForestClassifier
forest_model,forest_accuracy,forest_precision,forest_recall,forest_f1,forest_auc=train_and_evaluate(RandomForestClassifier(n_estimators=100,random_state=42))
results.append(["Forest model",forest_accuracy,forest_precision,forest_recall,forest_f1,forest_auc])

from sklearn.ensemble import GradientBoostingClassifier
gradient_model,gradient_accuracy,gradient_precision,gradient_recall,gradient_f1,gradient_auc=train_and_evaluate(GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42))
results.append(["Gradient model",gradient_accuracy,gradient_precision,gradient_recall,gradient_f1,gradient_auc])

from xgboost import XGBClassifier
xgb_model,xgb_accuracy,xgb_precision,xgb_recall,xgb_f1,xgb_auc=train_and_evaluate(XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42))
results.append(["XGB model",xgb_accuracy,xgb_precision,xgb_recall,xgb_f1,xgb_auc])

from sklearn.model_selection import GridSearchCV

logistic_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_grid = GridSearchCV(
    logistic_model,
    logistic_params,
    cv=5,
    scoring="f1"
)

logistic_grid.fit(X_train, y_train)

print("Best Logistic Params:", logistic_grid.best_params_)
print("Best Logistic CV Score:", logistic_grid.best_score_)

logistic_best = logistic_grid.best_estimator_

y_pred = logistic_best.predict(X_test)

print("Tuned Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

knn_params = {
    "model__n_neighbors": [3, 5, 7, 9],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"]
}

knn_grid = GridSearchCV(
    knn_model,
    knn_params,
    cv=5,
    scoring="f1"
)

knn_grid.fit(X_train, y_train)

print("Best KNN Params:", knn_grid.best_params_)
print("Best KNN CV Score:", knn_grid.best_score_)

knn_best = knn_grid.best_estimator_

knn_pred = knn_best.predict(X_test)

print("Tuned KNN")
print("Accuracy:", accuracy_score(y_test, knn_pred))
print("Precision:", precision_score(y_test, knn_pred))
print("Recall:", recall_score(y_test, knn_pred))
print("F1:", f1_score(y_test, knn_pred))
print("AUC:", roc_auc_score(y_test, knn_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, knn_pred))

tree_params = {
    "model__max_depth": [2, 3, 5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}
tree_grid = GridSearchCV(
    tree_model,
    tree_params,
    cv=5,
    scoring="f1"
)

tree_grid.fit(X_train, y_train)

print("Best Tree Params:", tree_grid.best_params_)
print("Best Tree CV Score:", tree_grid.best_score_)

tree_best = tree_grid.best_estimator_

tree_pred = tree_best.predict(X_test)

print("Tuned Decision Tree")
print("Accuracy:", accuracy_score(y_test, tree_pred))
print("Precision:", precision_score(y_test, tree_pred))
print("Recall:", recall_score(y_test, tree_pred))
print("F1:", f1_score(y_test, tree_pred))
print("AUC:", roc_auc_score(y_test, tree_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, tree_pred))

forest_params = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [None, 5],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

forest_grid = GridSearchCV(
    forest_model,
    forest_params,
    cv=5,
    scoring="f1"
)

forest_grid.fit(X_train, y_train)

print("Best Forest Params:", forest_grid.best_params_)
print("Best Forest CV Score:", forest_grid.best_score_)

forest_best = forest_grid.best_estimator_

forest_pred = forest_best.predict(X_test)

print("Tuned Random Forest")
print("Accuracy:", accuracy_score(y_test, forest_pred))
print("Precision:", precision_score(y_test, forest_pred))
print("Recall:", recall_score(y_test, forest_pred))
print("F1:", f1_score(y_test, forest_pred))
print("AUC:", roc_auc_score(y_test, forest_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, forest_pred))


gradient_params = {
    "model__n_estimators": [50, 100],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3]
}
gradient_grid = GridSearchCV(
    gradient_model,
    gradient_params,
    cv=5,
    scoring="f1"
)

gradient_grid.fit(X_train, y_train)

print("Best Gradient Params:", gradient_grid.best_params_)
print("Best Gradient CV Score:", gradient_grid.best_score_)

gradient_best = gradient_grid.best_estimator_

gradient_pred = gradient_best.predict(X_test)
gradient_prob = gradient_best.predict_proba(X_test)[:, 1]

print("Tuned Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, gradient_pred))
print("Precision:", precision_score(y_test, gradient_pred))
print("Recall:", recall_score(y_test, gradient_pred))
print("F1:", f1_score(y_test, gradient_pred))
print("ROC-AUC:", roc_auc_score(y_test, gradient_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_test, gradient_pred))
xgb_params = {
    "model__n_estimators": [50, 100],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    xgb_model,
    xgb_params,
    cv=5,
    scoring="f1"
)

xgb_grid.fit(X_train, y_train)

print("Best XGBoost Params:", xgb_grid.best_params_)
print("Best XGBoost CV Score:", xgb_grid.best_score_)

xgb_best = xgb_grid.best_estimator_

xgb_pred = xgb_best.predict(X_test)
xgb_prob = xgb_best.predict_proba(X_test)[:, 1]

print("Tuned XGBoost")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall:", recall_score(y_test, xgb_pred))
print("F1:", f1_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

LogisticRegression()
Accuracy:  0.6666666666666666
precision:  0.5
recall:  0.5
f1:  0.5
AUC score is:  0.75
Confusion matrix:  [[3 1]
 [1 1]]
------------------------------------------------------------
KNeighborsClassifier(n_neighbors=3)
Accuracy:  0.8333333333333334
precision:  0.6666666666666666
recall:  1.0
f1:  0.8
AUC score is:  0.9375
Confusion matrix:  [[3 1]
 [0 2]]
------------------------------------------------------------
DecisionTreeClassifier(random_state=42)
Accuracy:  1.0
precision:  1.0
recall:  1.0
f1:  1.0
AUC score is:  1.0
Confusion matrix:  [[4 0]
 [0 2]]
------------------------------------------------------------
RandomForestClassifier(random_state=42)
Accuracy:  0.6666666666666666
precision:  0.5
recall:  0.5
f1:  0.5
AUC score is:  0.875
Confusion matrix:  [[3 1]
 [1 1]]
------------------------------------------------------------
GradientBoostingClassifier(random_state=42)
Accuracy:  0.8333333333333334
precision:  0.6666666666666666
recall:  1.0
f1:  0.8
AU